# Chạy LLM extractor (Qwen2.5-7B) trên Google Colab
Self-host ≤9B, KHÔNG API ngoài. Runtime → Change runtime type → **GPU** (T4/L4/A100).
Mẹo: cache model vào Google Drive để khỏi tải lại 15GB mỗi phiên.

In [ ]:
# 1) (Khuyến nghị) Mount Drive + cache HuggingFace vào Drive để khỏi tải lại model
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'   # cache model bền vững
    print('HF_HOME =', os.environ['HF_HOME'])
except Exception as e:
    print('Bỏ qua Drive:', e)

In [ ]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

In [ ]:
# 3) Cài dependencies
!pip install -q -r requirements.txt
!pip install -q -r requirements-gpu.txt
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4) SMOKE TEST 3 file trước (kiểm model + JSON parse; lần đầu tải model ~15GB)
import os, shutil
os.makedirs('smoke/input', exist_ok=True)
for n in [1, 5, 50]:
    shutil.copy(f'data/test/input/{n}.txt', f'smoke/input/{n}.txt')
!python src/pipeline.py --input smoke/input --output smoke/out --backend llm
!cat smoke/out/5.json

In [ ]:
# 5) Chạy full 100 file (~1–3h với HF generate; dùng 3B để lặp nhanh nếu cần)
!python src/pipeline.py --input data/test/input --output output --backend llm

In [ ]:
# 6) Validate + đóng gói + tải về
!python scripts/package_submission.py --output output --input data/test/input --n 100
from google.colab import files; files.download('output.zip')

In [ ]:
# 7) Đo trên dev gold
!python src/eval/scorer.py --pred output --gold data/dev/gold --mode overlap
!python src/eval/eval_linking.py